# County-Year Vulnerability Dataset Building

## 1. Revised Dataset Setup

Prepare the existing Zillow-FEMA panel for the revised county-year vulnerability framework.

### 1.1 Imports and Project Paths

In [ ]:
# ==================================================
# Imports
# ==================================================

from pathlib import Path

import pandas as pd
import numpy as np

# ==================================================
# Project Paths
# ==================================================

PROJECT_ROOT = Path("..")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "merged_zillow_fema_data.csv"
)

RAW_DATA = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

PROCESSED_DATA = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

RESULTS_TABLES = (
    PROJECT_ROOT
    / "results"
    / "tables"
)

RESULTS_LOGS = (
    PROJECT_ROOT
    / "results"
    / "logs"
)

# create folders if needed
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)
RESULTS_TABLES.mkdir(parents=True, exist_ok=True)
RESULTS_LOGS.mkdir(parents=True, exist_ok=True)

# ==================================================
# Project Settings
# ==================================================

PROJECT_START_YEAR = 2011
PROJECT_END_YEAR = 2025

### 1.2 Load Existing Monthly Dataset

In [2]:
# ==================================================
# Load Existing Cleaned Zillow-FEMA Dataset
# ==================================================

df = pd.read_csv(DATA_PATH)

# convert date column to datetime
df["Date"] = pd.to_datetime(df["Date"])

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Number of counties:", df["STCOFIPS"].nunique())

df.head()

Dataset shape: (12767, 18)
Date range: 2010-01-31 00:00:00 to 2025-12-31 00:00:00
Number of counties: 67


,RegionID,SizeRank,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,Date,HousingPrice,STCOFIPS,STATE,STATEABBRV,COUNTY,POPULATION,CFLD_RISKS,HRCN_RISKS,SOVI_SCORE,RESL_SCORE
0,67,371,Bay County,FL,"Panama City, FL",12,5,2010-01-31,177323.256436,12005,Florida,FL,Bay,174869,77.2,98.664998,30.852417,29.452926
1,67,371,Bay County,FL,"Panama City, FL",12,5,2010-02-28,175726.799272,12005,Florida,FL,Bay,174869,77.2,98.664998,30.852417,29.452926
2,67,371,Bay County,FL,"Panama City, FL",12,5,2010-03-31,173648.913459,12005,Florida,FL,Bay,174869,77.2,98.664998,30.852417,29.452926
3,67,371,Bay County,FL,"Panama City, FL",12,5,2010-04-30,172490.101014,12005,Florida,FL,Bay,174869,77.2,98.664998,30.852417,29.452926
4,67,371,Bay County,FL,"Panama City, FL",12,5,2010-05-31,171749.827482,12005,Florida,FL,Bay,174869,77.2,98.664998,30.852417,29.452926


### 1.3 Dataset Validation

In [3]:
# ==================================================
# Dataset Validation
# ==================================================

num_rows = df.shape[0]
num_columns = df.shape[1]
num_counties = df["STCOFIPS"].nunique()
start_date = df["Date"].min()
end_date = df["Date"].max()

# duplicate county-month rows
duplicate_count = df.duplicated(
    subset=["STCOFIPS", "Date"]
).sum()

print("Rows:", num_rows)
print("Columns:", num_columns)
print("Counties:", num_counties)
print("Start Date:", start_date)
print("End Date:", end_date)
print("Duplicate county-month rows:", duplicate_count)

Rows: 12767
Columns: 18
Counties: 67
Start Date: 2010-01-31 00:00:00
End Date: 2025-12-31 00:00:00
Duplicate county-month rows: 0


In [4]:
# ==================================================
# Missing Value Check
# ==================================================

validation_cols = [
    "HousingPrice",
    "CFLD_RISKS",
    "HRCN_RISKS",
    "SOVI_SCORE",
    "RESL_SCORE",
    "POPULATION"
]

print("Missing Values:")
print(df[validation_cols].isnull().sum())

# ==================================================
# Negative Value Check
# ==================================================

numeric_checks = [
    "HousingPrice",
    "POPULATION"
]

print("\nNegative Values:")

for col in numeric_checks:
    negative_count = (df[col] < 0).sum()
    print(f"{col}: {negative_count}")

Missing Values:
HousingPrice    0
CFLD_RISKS      0
HRCN_RISKS      0
SOVI_SCORE      0
RESL_SCORE      0
POPULATION      0
dtype: int64

Negative Values:
HousingPrice: 0
POPULATION: 0


### 1.4 Key Takeaways

- The existing monthly Zillow-FEMA dataset is used as the input for the revised framework.
- The goal is to create annual housing-market indicators, not a direct price-prediction target.
- The county-year dataset will become the housing-market signal layer for vulnerability modeling.

## 2. County-Year Housing Market Indicators

Aggregate monthly Zillow values into annual county-level housing market signals.

### 2.1 Create Year Variable

In [5]:
# ==================================================
# Create Year Variable
# ==================================================

df["Year"] = df["Date"].dt.year

print("Years available:", df["Year"].min(), "to", df["Year"].max())
print("County-year combinations:", df[["STCOFIPS", "Year"]].drop_duplicates().shape[0])

Years available: 2010 to 2025
County-year combinations: 1064


### 2.2 Annual Housing Indicator Aggregation

In [7]:
# ==================================================
# County-Year Housing Indicator Aggregation
# ==================================================

# Handle missing Metro values consistently
df["Metro"] = df["Metro"].fillna("Non-Metro")

# Create metro indicator
df["is_metro"] = (df["Metro"] != "Non-Metro").astype(int)

id_cols = [
    "STCOFIPS",
    "RegionID",
    "RegionName",
    "State",
    "Metro",
    "StateCodeFIPS",
    "MunicipalCodeFIPS",
    "COUNTY"
]

county_year_housing = (
    df
    .groupby(id_cols + ["Year"], as_index=False)
    .agg(
        avg_annual_housing_price=("HousingPrice", "mean"),
        median_annual_housing_price=("HousingPrice", "median"),
        min_annual_housing_price=("HousingPrice", "min"),
        max_annual_housing_price=("HousingPrice", "max"),
        annual_price_volatility=("HousingPrice", "std"),
        monthly_observations=("HousingPrice", "count"),
        is_metro=("is_metro", "first"),
        SizeRank=("SizeRank", "first"),
        POPULATION=("POPULATION", "first"),
        CFLD_RISKS=("CFLD_RISKS", "first"),
        HRCN_RISKS=("HRCN_RISKS", "first"),
        SOVI_SCORE=("SOVI_SCORE", "first"),
        RESL_SCORE=("RESL_SCORE", "first")
    )
)

# Fill volatility for county-years with only one monthly observation
county_year_housing["annual_price_volatility"] = (
    county_year_housing["annual_price_volatility"]
    .fillna(0)
)

print("County-year housing shape:", county_year_housing.shape)
print("Counties:", county_year_housing["STCOFIPS"].nunique())
print("Years:", county_year_housing["Year"].min(), "to", county_year_housing["Year"].max())

county_year_housing.head()

County-year housing shape: (1064, 22)
Counties: 67
Years: 2010 to 2025


,STCOFIPS,RegionID,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,COUNTY,Year,avg_annual_housing_price,...,max_annual_housing_price,annual_price_volatility,monthly_observations,is_metro,SizeRank,POPULATION,CFLD_RISKS,HRCN_RISKS,SOVI_SCORE,RESL_SCORE
0,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2010,162126.833191,...,168775.803697,5833.729164,12,1,251,277984,0.0,96.704214,34.764631,80.979644
1,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2011,144811.606954,...,151065.304144,4024.282542,12,1,251,277984,0.0,96.704214,34.764631,80.979644
2,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2012,136832.426441,...,139626.338658,1809.886935,12,1,251,277984,0.0,96.704214,34.764631,80.979644
3,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2013,140015.149500,...,141791.385842,1500.697759,12,1,251,277984,0.0,96.704214,34.764631,80.979644
4,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2014,147354.521716,...,149219.273913,2125.835860,12,1,251,277984,0.0,96.704214,34.764631,80.979644


### 2.3 Annual Growth and Appreciation Features

In [8]:
# ==================================================
# Annual Growth and Appreciation Features
# ==================================================

county_year_housing = county_year_housing.sort_values(
    ["STCOFIPS", "Year"]
).reset_index(drop=True)

# previous-year average price
county_year_housing["prev_year_housing_price"] = (
    county_year_housing
    .groupby("STCOFIPS")["avg_annual_housing_price"]
    .shift(1)
)

# annual dollar growth
county_year_housing["annual_price_growth_dollar"] = (
    county_year_housing["avg_annual_housing_price"]
    - county_year_housing["prev_year_housing_price"]
)

# annual percentage growth
county_year_housing["annual_price_growth_pct"] = (
    county_year_housing["annual_price_growth_dollar"]
    / county_year_housing["prev_year_housing_price"]
    * 100
)

# baseline price for each county
county_year_housing["baseline_housing_price"] = (
    county_year_housing
    .groupby("STCOFIPS")["avg_annual_housing_price"]
    .transform("first")
)

# appreciation relative to first available year
county_year_housing["appreciation_from_baseline_pct"] = (
    (county_year_housing["avg_annual_housing_price"]
     - county_year_housing["baseline_housing_price"])
    / county_year_housing["baseline_housing_price"]
    * 100
)

# growth acceleration compared with previous annual growth rate
county_year_housing["price_growth_acceleration"] = (
    county_year_housing
    .groupby("STCOFIPS")["annual_price_growth_pct"]
    .diff()
)

county_year_housing.head()

,STCOFIPS,RegionID,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,COUNTY,Year,avg_annual_housing_price,...,CFLD_RISKS,HRCN_RISKS,SOVI_SCORE,RESL_SCORE,prev_year_housing_price,annual_price_growth_dollar,annual_price_growth_pct,baseline_housing_price,appreciation_from_baseline_pct,price_growth_acceleration
0,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2010,162126.833191,...,0.0,96.704214,34.764631,80.979644,NaN,NaN,NaN,162126.833191,0.000000,NaN
1,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2011,144811.606954,...,0.0,96.704214,34.764631,80.979644,162126.833191,-17315.226237,-10.680050,162126.833191,-10.680050,NaN
2,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2012,136832.426441,...,0.0,96.704214,34.764631,80.979644,144811.606954,-7979.180513,-5.510042,162126.833191,-15.601616,5.170008
3,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2013,140015.149500,...,0.0,96.704214,34.764631,80.979644,136832.426441,3182.723060,2.326001,162126.833191,-13.638510,7.836043
4,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2014,147354.521716,...,0.0,96.704214,34.764631,80.979644,140015.149500,7339.372216,5.241842,162126.833191,-9.111577,2.915841


### 2.4 Market Signal Flags

In [11]:
# ==================================================
# 2.4 Market Signal Flags
# ==================================================

# High annual growth threshold using valid growth values only
high_growth_threshold = county_year_housing["annual_price_growth_pct"].median()

# Create high-growth flag
# First-year rows have NaN growth, so keep the flag as NaN instead of forcing 0
county_year_housing["high_growth_flag"] = np.where(
    county_year_housing["annual_price_growth_pct"].isna(),
    np.nan,
    np.where(
        county_year_housing["annual_price_growth_pct"] >= high_growth_threshold,
        1,
        0
    )
)

# High volatility threshold using all county-years
high_volatility_threshold = county_year_housing["annual_price_volatility"].median()

# Create high-volatility flag
county_year_housing["high_volatility_flag"] = np.where(
    county_year_housing["annual_price_volatility"] >= high_volatility_threshold,
    1,
    0
)

# Complete-year flag based on monthly observations
county_year_housing["complete_year_flag"] = np.where(
    county_year_housing["monthly_observations"] == 12,
    1,
    0
)

print("High growth threshold:", round(high_growth_threshold, 2))
print("High volatility threshold:", round(high_volatility_threshold, 2))

county_year_housing[
    [
        "RegionName",
        "Year",
        "avg_annual_housing_price",
        "annual_price_growth_pct",
        "annual_price_volatility",
        "high_growth_flag",
        "high_volatility_flag",
        "complete_year_flag"
    ]
].head()

High growth threshold: 5.92
High volatility threshold: 3724.47


,RegionName,Year,avg_annual_housing_price,annual_price_growth_pct,annual_price_volatility,high_growth_flag,high_volatility_flag,complete_year_flag
0,Alachua County,2010,162126.833191,NaN,5833.729164,NaN,1,1
1,Alachua County,2011,144811.606954,-10.680050,4024.282542,0.0,1,1
2,Alachua County,2012,136832.426441,-5.510042,1809.886935,0.0,0,1
3,Alachua County,2013,140015.149500,2.326001,1500.697759,0.0,0,1
4,Alachua County,2014,147354.521716,5.241842,2125.835860,0.0,0,1


Section 2.5 — County-Year Housing Dataset Validation.

In [12]:
# ==================================================
# 2.5 County-Year Housing Dataset Validation
# ==================================================

print("County-year dataset shape:", county_year_housing.shape)
print("Unique counties:", county_year_housing["STCOFIPS"].nunique())
print("Year range:", county_year_housing["Year"].min(), "to", county_year_housing["Year"].max())

print("\nMonthly observation coverage:")
print(county_year_housing["monthly_observations"].value_counts().sort_index())

print("\nComplete-year coverage:")
print(county_year_housing["complete_year_flag"].value_counts())

print("\nMissing values:")
missing_summary = (
    county_year_housing
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[missing_summary > 0]

County-year dataset shape: (1064, 31)
Unique counties: 67
Year range: 2010 to 2025

Monthly observation coverage:
monthly_observations
11       1
12    1063
Name: count, dtype: int64

Complete-year coverage:
complete_year_flag
1    1063
0       1
Name: count, dtype: int64

Missing values:


price_growth_acceleration     134
prev_year_housing_price        67
annual_price_growth_dollar     67
high_growth_flag               67
annual_price_growth_pct        67
dtype: int64

The county-year housing dataset contains all 67 Florida counties from 2010–2025. Monthly coverage is nearly complete, with only one county-year containing 11 observations instead of 12. Missing values appear only in lagged growth and acceleration features, which is expected because the first year has no previous-year comparison and acceleration requires two years of growth history.

## 3. Final County-Year Housing Layer

Filter the annual dataset to the project period and validate the housing-market signal layer.

### 3.1 Filter to Project Period

In [13]:
# ==================================================
# Filter to Project Period
# ==================================================

county_year_housing_final = county_year_housing[
    (county_year_housing["Year"] >= PROJECT_START_YEAR)
    & (county_year_housing["Year"] <= PROJECT_END_YEAR)
].copy()

print("Final county-year housing shape:", county_year_housing_final.shape)
print("Counties:", county_year_housing_final["STCOFIPS"].nunique())
print("Years:", county_year_housing_final["Year"].min(), "to", county_year_housing_final["Year"].max())
print("Duplicate county-year rows:", county_year_housing_final.duplicated(subset=["STCOFIPS", "Year"]).sum())

county_year_housing_final.head()

Final county-year housing shape: (1000, 31)
Counties: 67
Years: 2011 to 2025
Duplicate county-year rows: 0


,STCOFIPS,RegionID,RegionName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,COUNTY,Year,avg_annual_housing_price,...,RESL_SCORE,prev_year_housing_price,annual_price_growth_dollar,annual_price_growth_pct,baseline_housing_price,appreciation_from_baseline_pct,price_growth_acceleration,high_growth_flag,high_volatility_flag,complete_year_flag
1,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2011,144811.606954,...,80.979644,162126.833191,-17315.226237,-10.680050,162126.833191,-10.680050,NaN,0.0,1,1
2,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2012,136832.426441,...,80.979644,144811.606954,-7979.180513,-5.510042,162126.833191,-15.601616,5.170008,0.0,0,1
3,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2013,140015.149500,...,80.979644,136832.426441,3182.723060,2.326001,162126.833191,-13.638510,7.836043,0.0,0,1
4,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2014,147354.521716,...,80.979644,140015.149500,7339.372216,5.241842,162126.833191,-9.111577,2.915841,0.0,0,1
5,12001,1509,Alachua County,FL,"Gainesville, FL",12,1,Alachua,2015,153750.122064,...,80.979644,147354.521716,6395.600348,4.340281,162126.833191,-5.166764,-0.901560,0.0,0,1


### 3.2 County-Year Dataset Overview

In [14]:
# ==================================================
# County-Year Dataset Overview Table
# ==================================================

county_year_overview = pd.DataFrame({
    "Metric": [
        "Observations",
        "Variables",
        "Counties",
        "Start Year",
        "End Year",
        "Duplicate County-Year Rows",
        "Complete County-Year Rows"
    ],
    "Value": [
        county_year_housing_final.shape[0],
        county_year_housing_final.shape[1],
        county_year_housing_final["STCOFIPS"].nunique(),
        county_year_housing_final["Year"].min(),
        county_year_housing_final["Year"].max(),
        county_year_housing_final.duplicated(subset=["STCOFIPS", "Year"]).sum(),
        int(county_year_housing_final["complete_year_flag"].sum())
    ]
})

# display table
display(county_year_overview)

# save table
county_year_overview.to_csv(
    RESULTS_TABLES / "county_year_housing_overview.csv",
    index=False
)

print(
    f"Saved: "
    f"{RESULTS_TABLES / 'county_year_housing_overview.csv'}"
)

,Metric,Value
0,Observations,1000
1,Variables,31
2,Counties,67
3,Start Year,2011
4,End Year,2025
5,Duplicate County-Year Rows,0
6,Complete County-Year Rows,999


Saved: ..\results\tables\county_year_housing_overview.csv


### 3.3 Annual Housing Summary

In [15]:
# ==================================================
# Annual Housing Summary Table
# ==================================================

annual_housing_summary = (
    county_year_housing_final
    .groupby("Year", as_index=False)
    .agg(
        avg_florida_housing_price=("avg_annual_housing_price", "mean"),
        median_florida_housing_price=("avg_annual_housing_price", "median"),
        avg_annual_growth_pct=("annual_price_growth_pct", "mean"),
        avg_annual_volatility=("annual_price_volatility", "mean"),
        county_count=("STCOFIPS", "nunique")
    )
    .round(2)
)

# display table
display(annual_housing_summary.head())

# save table
annual_housing_summary.to_csv(
    RESULTS_TABLES / "annual_housing_summary.csv",
    index=False
)

print(
    f"Saved: "
    f"{RESULTS_TABLES / 'annual_housing_summary.csv'}"
)

,Year,avg_florida_housing_price,median_florida_housing_price,avg_annual_growth_pct,avg_annual_volatility,county_count
0,2011,123820.62,116077.90,-7.58,2628.04,66
1,2012,122216.85,114642.82,-1.33,1760.46,66
2,2013,131230.99,125456.69,6.78,4036.37,66
3,2014,142503.33,134473.35,7.83,2901.59,66
4,2015,152479.77,145364.55,6.58,4011.38,66


Saved: ..\results\tables\annual_housing_summary.csv


### 3.4 Missing Value Check

In [16]:
# ==================================================
# Final Missing Value Check
# ==================================================

final_check_cols = [
    "avg_annual_housing_price",
    "annual_price_growth_pct",
    "annual_price_volatility",
    "appreciation_from_baseline_pct",
    "CFLD_RISKS",
    "HRCN_RISKS",
    "SOVI_SCORE",
    "RESL_SCORE",
    "POPULATION"
]

missing_summary = (
    county_year_housing_final[final_check_cols]
    .isnull()
    .sum()
    .reset_index()
    .rename(columns={"index": "Variable", 0: "Missing Values"})
)

missing_summary

,Variable,Missing Values
0,avg_annual_housing_price,0
1,annual_price_growth_pct,3
2,annual_price_volatility,0
3,appreciation_from_baseline_pct,0
4,CFLD_RISKS,0
5,HRCN_RISKS,0
6,SOVI_SCORE,0
7,RESL_SCORE,0
8,POPULATION,0


### 3.5 Save County-Year Housing Layer

In [17]:
# ==================================================
# Save County-Year Housing Layer
# ==================================================

output_path = (
    PROCESSED_DATA
    / "county_year_housing_indicators.csv"
)

county_year_housing_final.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")
print("Final shape:", county_year_housing_final.shape)

Saved: ..\data\processed\county_year_housing_indicators.csv
Final shape: (1000, 31)


### Key Takeaways

- Monthly Zillow housing values were converted into county-year housing-market indicators.
- Housing price is now treated as a market signal layer rather than the only prediction target.
- The saved dataset will be used next for spatial features, disaster history, and socioeconomic indicators.

## 4. Next Step

Prepare county adjacency and neighboring-county housing features using the Florida county shapefile.

## 4. Spatial Feature Preparation

### 4.1 Load Florida County Shapefile

Load the Florida county boundary shapefile so spatial relationships between counties can be created.

In [23]:
# ==================================================
# 4.1 Load Florida County Shapefile
# ==================================================

import geopandas as gpd

# path to county shapefile
county_shapefile_path = (
    RAW_DATA
    / "shapefiles"
    / "tl_2025_us_county"
    / "tl_2025_us_county.shp"
)

# load county shapefile
counties_gdf = gpd.read_file(county_shapefile_path)

print("Original shapefile shape:", counties_gdf.shape)
print("Original CRS:", counties_gdf.crs)

counties_gdf.head()

Original shapefile shape: (3235, 19)
Original CRS: EPSG:4269


,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,40,075,01101825,40075,0500000US40075,Kiowa,Kiowa County,06,H1,G4020,NaN,NaN,NaN,A,2629039892,40296743,+34.9214893,-098.9816168,"POLYGON ((-98.95506 35.11643, -98.94903 35.116..."
1,46,079,01265776,46079,0500000US46079,Lake,Lake County,06,H1,G4020,NaN,NaN,NaN,A,1457916151,31746795,+44.0284497,-097.1232229,"POLYGON ((-96.88886 43.9353, -96.88886 43.9351..."
2,37,033,01008542,37033,0500000US37033,Caswell,Caswell County,06,H1,G4020,NaN,NaN,NaN,A,1102042927,8293623,+36.3943252,-079.3396193,"POLYGON ((-79.14343 36.4422, -79.14345 36.4418..."
3,48,377,01383974,48377,0500000US48377,Presidio,Presidio County,06,H1,G4020,NaN,NaN,NaN,A,9985057447,1773188,+30.0058912,-104.2616192,"POLYGON ((-104.98078 30.62552, -104.98073 30.6..."
4,39,057,01074041,39057,0500000US39057,Greene,Greene County,06,H1,G4020,212,19430,NaN,A,1071302625,6798109,+39.6874785,-083.8948943,"POLYGON ((-84.10668 39.68891, -84.10662 39.689..."


In [24]:
# ==================================================
# Filter to Florida Counties
# ==================================================

florida_counties_gdf = counties_gdf[
    counties_gdf["STATEFP"] == "12"
].copy()

print("Florida county shapefile shape:", florida_counties_gdf.shape)
print("Florida counties:", florida_counties_gdf["COUNTYFP"].nunique())

florida_counties_gdf[
    ["STATEFP", "COUNTYFP", "GEOID", "NAME", "NAMELSAD", "geometry"]
].head()

Florida county shapefile shape: (67, 19)
Florida counties: 67


,STATEFP,COUNTYFP,GEOID,NAME,NAMELSAD,geometry
22,12,045,12045,Gulf,Gulf County,"POLYGON ((-85.39126 30.02816, -85.39123 30.028..."
46,12,105,12105,Polk,Polk County,"POLYGON ((-81.42455 28.01248, -81.42455 28.012..."
93,12,079,12079,Madison,Madison County,"POLYGON ((-83.82191 30.30749, -83.8219 30.3076..."
173,12,029,12029,Dixie,Dixie County,"MULTIPOLYGON (((-82.91871 29.82408, -82.91672 ..."
187,12,097,12097,Osceola,Osceola County,"POLYGON ((-81.65727 28.3471, -81.65666 28.3471..."


### 4.2 Validate County Coverage

Check whether the Florida shapefile counties match the county identifiers in the county-year housing dataset.

In [25]:
# ==================================================
# 4.2 Validate County Coverage
# ==================================================

# Make shapefile county ID consistent with housing dataset county ID
florida_counties_gdf["STCOFIPS"] = florida_counties_gdf["GEOID"].astype(str)

county_year_housing_final["STCOFIPS"] = (
    county_year_housing_final["STCOFIPS"]
    .astype(str)
    .str.zfill(5)
)

# unique county IDs
shapefile_counties = set(florida_counties_gdf["STCOFIPS"].unique())
housing_counties = set(county_year_housing_final["STCOFIPS"].unique())

# compare coverage
missing_in_shapefile = sorted(housing_counties - shapefile_counties)
missing_in_housing = sorted(shapefile_counties - housing_counties)

print("Counties in shapefile:", len(shapefile_counties))
print("Counties in housing dataset:", len(housing_counties))
print("Missing in shapefile:", missing_in_shapefile)
print("Missing in housing dataset:", missing_in_housing)

Counties in shapefile: 67
Counties in housing dataset: 67
Missing in shapefile: []
Missing in housing dataset: []


In [26]:
# ==================================================
# Matched County Name Check
# ==================================================

county_match_check = (
    florida_counties_gdf[["STCOFIPS", "NAME", "NAMELSAD"]]
    .merge(
        county_year_housing_final[["STCOFIPS", "RegionName", "COUNTY"]]
        .drop_duplicates(),
        on="STCOFIPS",
        how="inner"
    )
    .sort_values("STCOFIPS")
)

print("Matched counties:", county_match_check["STCOFIPS"].nunique())

county_match_check.head()

Matched counties: 67


,STCOFIPS,NAME,NAMELSAD,RegionName,COUNTY
63,12001,Alachua,Alachua County,Alachua County,Alachua
22,12003,Baker,Baker County,Baker County,Baker
8,12005,Bay,Bay County,Bay County,Bay
42,12007,Bradford,Bradford County,Bradford County,Bradford
32,12009,Brevard,Brevard County,Brevard County,Brevard


### 4.3 Create County Adjacency Table

Identify neighboring Florida counties using shared county boundaries.

In [27]:
# ==================================================
# 4.3 Create County Adjacency Table
# ==================================================

# keep only needed columns
county_geometries = florida_counties_gdf[
    ["STCOFIPS", "NAME", "NAMELSAD", "geometry"]
].copy()

# create spatial join to identify touching counties
county_neighbors = gpd.sjoin(
    county_geometries,
    county_geometries,
    how="inner",
    predicate="touches",
    lsuffix="county",
    rsuffix="neighbor"
)

# remove self-matches if any
county_neighbors = county_neighbors[
    county_neighbors["STCOFIPS_county"] != county_neighbors["STCOFIPS_neighbor"]
].copy()

# clean adjacency table
county_adjacency = county_neighbors[
    [
        "STCOFIPS_county",
        "NAME_county",
        "STCOFIPS_neighbor",
        "NAME_neighbor"
    ]
].rename(
    columns={
        "STCOFIPS_county": "county_fips",
        "NAME_county": "county_name",
        "STCOFIPS_neighbor": "neighbor_fips",
        "NAME_neighbor": "neighbor_name"
    }
).sort_values(
    ["county_fips", "neighbor_fips"]
).reset_index(drop=True)

print("County adjacency rows:", county_adjacency.shape[0])
print("Counties with neighbors:", county_adjacency["county_fips"].nunique())

county_adjacency.head(10)

County adjacency rows: 320
Counties with neighbors: 67


,county_fips,county_name,neighbor_fips,neighbor_name
0,12001,Alachua,12007,Bradford
1,12001,Alachua,12023,Columbia
2,12001,Alachua,12041,Gilchrist
3,12001,Alachua,12075,Levy
4,12001,Alachua,12083,Marion
5,12001,Alachua,12107,Putnam
6,12001,Alachua,12125,Union
7,12003,Baker,12007,Bradford
8,12003,Baker,12019,Clay
9,12003,Baker,12023,Columbia


In [28]:
# ==================================================
# Neighbor Count Validation
# ==================================================

neighbor_counts = (
    county_adjacency
    .groupby(["county_fips", "county_name"], as_index=False)
    .agg(neighbor_count=("neighbor_fips", "nunique"))
    .sort_values("neighbor_count")
)

print("Minimum neighbors:", neighbor_counts["neighbor_count"].min())
print("Maximum neighbors:", neighbor_counts["neighbor_count"].max())
print("Average neighbors:", round(neighbor_counts["neighbor_count"].mean(), 2))

neighbor_counts.head(10)

Minimum neighbors: 1
Maximum neighbors: 10
Average neighbors: 4.78


,county_fips,county_name,neighbor_count
15,12033,Escambia,1
43,12087,Monroe,2
45,12091,Okaloosa,2
56,12113,Santa Rosa,2
51,12103,Pinellas,2
44,12089,Nassau,2
17,12037,Franklin,3
22,12047,Hamilton,3
42,12086,Miami-Dade,3
55,12111,St. Lucie,3


In [29]:
# ==================================================
# Save County Adjacency Table
# ==================================================

county_adjacency.to_csv(
    PROCESSED_DATA / "florida_county_adjacency.csv",
    index=False
)

print(
    f"Saved: "
    f"{PROCESSED_DATA / 'florida_county_adjacency.csv'}"
)

Saved: ..\data\processed\florida_county_adjacency.csv


### 4.4 Create Neighboring County Housing Features

Use the county adjacency table to calculate neighboring-county housing market indicators for each county-year.

In [30]:
# ==================================================
# 4.4 Create Neighboring County Housing Features
# ==================================================

# Keep housing variables needed for neighboring-county features
neighbor_housing_base = county_year_housing_final[
    [
        "STCOFIPS",
        "Year",
        "avg_annual_housing_price",
        "annual_price_growth_pct",
        "annual_price_volatility",
        "high_growth_flag",
        "high_volatility_flag"
    ]
].copy()

# Rename columns so they represent neighbor values
neighbor_housing_base = neighbor_housing_base.rename(
    columns={
        "STCOFIPS": "neighbor_fips",
        "avg_annual_housing_price": "neighbor_housing_price",
        "annual_price_growth_pct": "neighbor_price_growth_pct",
        "annual_price_volatility": "neighbor_price_volatility",
        "high_growth_flag": "neighbor_high_growth_flag",
        "high_volatility_flag": "neighbor_high_volatility_flag"
    }
)

# Merge adjacency table with neighbor housing values by neighbor county and year
county_neighbor_housing = county_adjacency.merge(
    neighbor_housing_base,
    on="neighbor_fips",
    how="left"
)

county_neighbor_housing.head()

,county_fips,county_name,neighbor_fips,neighbor_name,Year,neighbor_housing_price,neighbor_price_growth_pct,neighbor_price_volatility,neighbor_high_growth_flag,neighbor_high_volatility_flag
0,12001,Alachua,12007,Bradford,2011,102337.066909,-5.361393,995.247406,0.0,0
1,12001,Alachua,12007,Bradford,2012,101934.137211,-0.393728,1473.306233,0.0,0
2,12001,Alachua,12007,Bradford,2013,102435.128908,0.491486,1273.026499,0.0,0
3,12001,Alachua,12007,Bradford,2014,105777.819611,3.263227,955.652456,0.0,0
4,12001,Alachua,12007,Bradford,2015,106147.061966,0.349074,2059.993406,0.0,0


In [31]:
# ==================================================
# Aggregate Neighboring Housing Features
# ==================================================

neighbor_housing_features = (
    county_neighbor_housing
    .groupby(["county_fips", "Year"], as_index=False)
    .agg(
        neighbor_count=("neighbor_fips", "nunique"),
        neighbor_avg_housing_price=("neighbor_housing_price", "mean"),
        neighbor_avg_price_growth_pct=("neighbor_price_growth_pct", "mean"),
        neighbor_avg_price_volatility=("neighbor_price_volatility", "mean"),
        neighbor_high_growth_share=("neighbor_high_growth_flag", "mean"),
        neighbor_high_volatility_share=("neighbor_high_volatility_flag", "mean")
    )
)

neighbor_housing_features = neighbor_housing_features.rename(
    columns={
        "county_fips": "STCOFIPS"
    }
)

print("Neighbor housing feature shape:", neighbor_housing_features.shape)
print("Counties:", neighbor_housing_features["STCOFIPS"].nunique())
print("Years:", neighbor_housing_features["Year"].min(), "to", neighbor_housing_features["Year"].max())

neighbor_housing_features.head()

Neighbor housing feature shape: (1005, 8)
Counties: 67
Years: 2011 to 2025


,STCOFIPS,Year,neighbor_count,neighbor_avg_housing_price,neighbor_avg_price_growth_pct,neighbor_avg_price_volatility,neighbor_high_growth_share,neighbor_high_volatility_share
0,12001,2011,7,102374.849631,-6.763815,1794.893064,0.000000,0.142857
1,12001,2012,7,99827.485576,-2.455404,1117.279737,0.000000,0.000000
2,12001,2013,7,102229.301664,2.368021,1412.403491,0.142857,0.000000
3,12001,2014,7,106775.555497,4.331896,1637.672940,0.142857,0.000000
4,12001,2015,7,111968.466397,4.816762,3004.397627,0.428571,0.142857


In [32]:
# ==================================================
# Neighbor Housing Feature Missing Value Check
# ==================================================

neighbor_housing_features.isna().sum()

STCOFIPS                          0
Year                              0
neighbor_count                    0
neighbor_avg_housing_price        0
neighbor_avg_price_growth_pct     0
neighbor_avg_price_volatility     0
neighbor_high_growth_share        0
neighbor_high_volatility_share    0
dtype: int64

In [33]:
# ==================================================
# Merge Neighboring Features with County-Year Housing Data
# ==================================================

county_year_housing_spatial = county_year_housing_final.merge(
    neighbor_housing_features,
    on=["STCOFIPS", "Year"],
    how="left"
)

print("County-year housing with spatial features:", county_year_housing_spatial.shape)
print("Counties:", county_year_housing_spatial["STCOFIPS"].nunique())
print("Years:", county_year_housing_spatial["Year"].min(), "to", county_year_housing_spatial["Year"].max())
print("Duplicate county-year rows:", county_year_housing_spatial.duplicated(subset=["STCOFIPS", "Year"]).sum())

county_year_housing_spatial[
    [
        "RegionName",
        "Year",
        "avg_annual_housing_price",
        "annual_price_growth_pct",
        "neighbor_count",
        "neighbor_avg_housing_price",
        "neighbor_avg_price_growth_pct",
        "neighbor_high_growth_share"
    ]
].head()

County-year housing with spatial features: (1000, 37)
Counties: 67
Years: 2011 to 2025
Duplicate county-year rows: 0


,RegionName,Year,avg_annual_housing_price,annual_price_growth_pct,neighbor_count,neighbor_avg_housing_price,neighbor_avg_price_growth_pct,neighbor_high_growth_share
0,Alachua County,2011,144811.606954,-10.680050,7,102374.849631,-6.763815,0.000000
1,Alachua County,2012,136832.426441,-5.510042,7,99827.485576,-2.455404,0.000000
2,Alachua County,2013,140015.149500,2.326001,7,102229.301664,2.368021,0.142857
3,Alachua County,2014,147354.521716,5.241842,7,106775.555497,4.331896,0.142857
4,Alachua County,2015,153750.122064,4.340281,7,111968.466397,4.816762,0.428571


In [34]:
# ==================================================
# Spatial Housing Feature Validation
# ==================================================

spatial_feature_cols = [
    "neighbor_count",
    "neighbor_avg_housing_price",
    "neighbor_avg_price_growth_pct",
    "neighbor_avg_price_volatility",
    "neighbor_high_growth_share",
    "neighbor_high_volatility_share"
]

print("Missing values in spatial features:")
print(county_year_housing_spatial[spatial_feature_cols].isna().sum())

print("\nNeighbor count summary:")
print(county_year_housing_spatial["neighbor_count"].describe())

Missing values in spatial features:
neighbor_count                    0
neighbor_avg_housing_price        0
neighbor_avg_price_growth_pct     0
neighbor_avg_price_volatility     0
neighbor_high_growth_share        0
neighbor_high_volatility_share    0
dtype: int64

Neighbor count summary:
count    1000.000000
mean        4.780000
std         1.727595
min         1.000000
25%         4.000000
50%         5.000000
75%         6.000000
max        10.000000
Name: neighbor_count, dtype: float64


### 4.5 Save Spatial Housing Feature Layer

Save the county-year housing dataset with neighboring-county spatial features for later vulnerability modelling.

In [35]:
# ==================================================
# 4.5 Save Spatial Housing Feature Layer
# ==================================================

county_year_housing_spatial.to_csv(
    PROCESSED_DATA / "county_year_housing_spatial_features.csv",
    index=False
)

neighbor_housing_features.to_csv(
    PROCESSED_DATA / "neighbor_housing_features.csv",
    index=False
)

print(
    f"Saved: "
    f"{PROCESSED_DATA / 'county_year_housing_spatial_features.csv'}"
)

print(
    f"Saved: "
    f"{PROCESSED_DATA / 'neighbor_housing_features.csv'}"
)

Saved: ..\data\processed\county_year_housing_spatial_features.csv
Saved: ..\data\processed\neighbor_housing_features.csv
